# 00. ESS 배터리 데이터 준비

대용량 MATLAB v7.3 파일에서 DAY 1에 필요한 초기 10·100사이클 정보만 추출합니다.

- 원본 파일은 읽기 전용으로 사용합니다.
- 피처 선택은 Batch 1에서만 수행합니다.
- Batch 2는 동일한 변환 규칙을 적용하되 최종 평가 전까지 타깃 기반 선택에 사용하지 않습니다.

In [ ]:
from pathlib import Path
import re
import warnings

import h5py
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'archive'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

BATCH_FILES = {
    'batch1': RAW_DIR / '2017-05-12_batchdata_updated_struct_errorcorrect.mat',
    'batch2': RAW_DIR / '2018-02-20_batchdata_updated_struct_errorcorrect.mat',
}

for name, path in BATCH_FILES.items():
    print(f'{name}: {path.name} / {path.stat().st_size / 1024**3:.2f} GB')

## 파일 구조 확인

전체 배열을 출력하지 않고 그룹명, shape, dtype만 확인합니다.

In [ ]:
def inspect_mat(path):
    with h5py.File(path, 'r') as f:
        print('top-level:', list(f.keys()))
        batch = f['batch']
        print('cell count:', batch['cycle_life'].shape[0])
        for field in batch.keys():
            print(f'{field:16s} shape={batch[field].shape}, dtype={batch[field].dtype}')

inspect_mat(BATCH_FILES['batch1'])

## 추출 함수

`cycle_life`, 충전 조건, 초기 요약값, 10→100사이클 변화량, ΔQ(V) 통계량을 셀 단위로 만듭니다.

In [ ]:
def deref(handle, ref):
    return handle[ref]


def numeric_array(obj):
    return np.asarray(obj[()]).squeeze()


def scalar_from_ref(handle, ref):
    value = numeric_array(deref(handle, ref))
    return float(np.ravel(value)[0])


def decode_matlab_char(obj):
    arr = np.asarray(obj[()]).squeeze()
    if arr.dtype.kind in 'ui':
        if arr.size and np.nanmax(arr) > 0x10FFFF:
            return ''
        return ''.join(chr(int(x)) for x in np.ravel(arr) if int(x) != 0)
    if arr.dtype.kind in 'SU':
        return ''.join(str(x) for x in np.ravel(arr))
    return str(arr)


def string_from_ref(handle, ref):
    return decode_matlab_char(deref(handle, ref)).strip()


def get_group_vector(group, key):
    if key not in group:
        return None
    return np.asarray(group[key][()]).squeeze().astype(float)


def nearest_cycle_index(cycle_numbers, target):
    cycle_numbers = np.asarray(cycle_numbers, dtype=float).ravel()
    if cycle_numbers.size == 0 or np.all(~np.isfinite(cycle_numbers)):
        return None
    return int(np.nanargmin(np.abs(cycle_numbers - target)))


def cycle_curve(handle, cycles_group, field, index):
    refs = cycles_group[field]
    ref = refs[index, 0] if refs.shape[0] > index else refs[0, index]
    return numeric_array(deref(handle, ref)).astype(float).ravel()


def parse_policy(policy):
    nums = [float(x) for x in re.findall(r'([0-9]+(?:\.[0-9]+)?)', policy or '')]
    return {
        'c_rate_1': nums[0] if len(nums) > 0 else np.nan,
        'switch_soc': nums[1] if len(nums) > 1 else np.nan,
        'c_rate_2': nums[2] if len(nums) > 2 else np.nan,
    }


def safe_summary_value(summary, field, index):
    values = get_group_vector(summary, field)
    if values is None or values.size <= index:
        return np.nan
    return float(values[index])


def extract_batch(path, batch_name):
    rows, dq_curves, voltage_axes = [], [], []
    degradation_cycles, degradation_qd = [], []

    with h5py.File(path, 'r') as f:
        batch = f['batch']
        n_cells = batch['cycle_life'].shape[0]

        for cell_idx in range(n_cells):
            try:
                cycle_life = scalar_from_ref(f, batch['cycle_life'][cell_idx, 0])
                policy = string_from_ref(f, batch['policy_readable'][cell_idx, 0])
                if not policy:
                    policy = string_from_ref(f, batch['policy'][cell_idx, 0])

                barcode = string_from_ref(f, batch['barcode'][cell_idx, 0])
                summary = deref(f, batch['summary'][cell_idx, 0])
                cycles = deref(f, batch['cycles'][cell_idx, 0])
                vdlin = numeric_array(deref(f, batch['Vdlin'][cell_idx, 0])).astype(float).ravel()

                cycle_numbers = get_group_vector(summary, 'cycle')
                if cycle_numbers is None:
                    candidate = next((get_group_vector(summary, k) for k in summary.keys()), None)
                    cycle_numbers = np.arange(len(candidate)) + 1

                idx10 = nearest_cycle_index(cycle_numbers, 10)
                idx100 = nearest_cycle_index(cycle_numbers, 100)
                actual10 = float(cycle_numbers[idx10])
                actual100 = float(cycle_numbers[idx100])

                row = {
                    'batch': batch_name,
                    'cell_id': f'{batch_name}_cell_{cell_idx:02d}',
                    'barcode': barcode,
                    'charging_policy': policy,
                    'cycle_life': cycle_life,
                    'cycle_10_actual': actual10,
                    'cycle_100_actual': actual100,
                }
                row.update(parse_policy(policy))

                qd_history = get_group_vector(summary, 'QD')
                if qd_history is None:
                    qd_history = get_group_vector(summary, 'QDischarge')
                hist_n = min(len(cycle_numbers), len(qd_history))
                degradation_cycles.append(np.asarray(cycle_numbers[:hist_n], dtype=float))
                degradation_qd.append(np.asarray(qd_history[:hist_n], dtype=float))

                summary_fields = {
                    'QD': ['QD', 'QDischarge'],
                    'QC': ['QC', 'QCharge'],
                    'IR': ['IR'],
                    'Tavg': ['Tavg'],
                    'Tmin': ['Tmin'],
                    'Tmax': ['Tmax'],
                    'chargetime': ['chargetime', 'chargeTime'],
                }
                for output_name, candidates in summary_fields.items():
                    source = next((x for x in candidates if x in summary), None)
                    v10 = safe_summary_value(summary, source, idx10) if source else np.nan
                    v100 = safe_summary_value(summary, source, idx100) if source else np.nan
                    row[f'{output_name}_10'] = v10
                    row[f'{output_name}_100'] = v100
                    row[f'{output_name}_diff'] = v100 - v10
                    denom = actual100 - actual10
                    row[f'{output_name}_slope'] = (v100 - v10) / denom if denom else np.nan
                    row[f'{output_name}_ratio'] = v100 / v10 if np.isfinite(v10) and v10 != 0 else np.nan

                q10 = cycle_curve(f, cycles, 'Qdlin', idx10)
                q100 = cycle_curve(f, cycles, 'Qdlin', idx100)
                n = min(len(vdlin), len(q10), len(q100))
                voltage = vdlin[:n]
                dq = q100[:n] - q10[:n]
                finite = np.isfinite(voltage) & np.isfinite(dq)
                voltage, dq = voltage[finite], dq[finite]
                order = np.argsort(voltage)
                voltage, dq = voltage[order], dq[order]

                if dq.size:
                    row.update({
                        'dQ_mean': float(np.mean(dq)),
                        'dQ_std': float(np.std(dq, ddof=1)),
                        'dQ_var': float(np.var(dq, ddof=1)),
                        'dQ_log_var': float(np.log10(max(np.var(dq, ddof=1), 1e-12))),
                        'dQ_min': float(np.min(dq)),
                        'dQ_max': float(np.max(dq)),
                        'dQ_abs_area': float(np.trapezoid(np.abs(dq), voltage)),
                        'dQ_area': float(np.trapezoid(dq, voltage)),
                        'dQ_min_voltage': float(voltage[np.argmin(dq)]),
                    })
                dq_curves.append(dq)
                voltage_axes.append(voltage)
                rows.append(row)
                print(f'[{batch_name}] {cell_idx + 1:02d}/{n_cells}: {row["cell_id"]}', flush=True)
            except Exception as exc:
                print(f'[{batch_name}] cell {cell_idx} skipped: {type(exc).__name__}: {exc}', flush=True)

    frame = pd.DataFrame(rows)
    frame.to_parquet(PROCESSED_DIR / f'{batch_name}_features.parquet', index=False)
    np.savez_compressed(
        PROCESSED_DIR / f'{batch_name}_dq_curves.npz',
        dq=np.array(dq_curves, dtype=object),
        voltage=np.array(voltage_axes, dtype=object),
    )
    np.savez_compressed(
        PROCESSED_DIR / f'{batch_name}_degradation_curves.npz',
        cycle=np.array(degradation_cycles, dtype=object),
        qd=np.array(degradation_qd, dtype=object),
    )
    return frame

## Batch 1 추출 실행

DAY 1 피처 탐색에 사용할 Batch 1을 먼저 추출합니다.

In [ ]:
batch1 = extract_batch(BATCH_FILES['batch1'], 'batch1')
print('shape:', batch1.shape)
display(batch1.head())
display(batch1[['cycle_life', 'QD_10', 'QD_100', 'dQ_log_var']].describe())

## Batch 2 변환 준비

Batch 2에는 같은 함수를 적용합니다. DAY 1에서는 피처 선택에 Batch 2 타깃을 사용하지 않습니다.

In [ ]:
RUN_BATCH2_EXTRACTION = False

if RUN_BATCH2_EXTRACTION:
    batch2 = extract_batch(BATCH_FILES['batch2'], 'batch2')
    print('shape:', batch2.shape)

## 데이터 품질 확인

In [ ]:
quality = pd.DataFrame({
    'dtype': batch1.dtypes.astype(str),
    'missing_n': batch1.isna().sum(),
    'missing_rate': batch1.isna().mean(),
    'unique_n': batch1.nunique(dropna=True),
}).sort_values(['missing_rate', 'unique_n'], ascending=[False, True])
display(quality)